# Tutorial for Training/Deploying the Physical CartPole Demo

This notebook follows the demo-hls4ml-25 README (Steps 1–7)

> Use the navigation links below to jump around quickly.


## Navigation
- [Step 1 — Environment Setup](#step-1--environment-setup)
- [Step 2 — Training Neural Network Controller](#step-2--training-neural-network-controller)
<!-- - [Step 3 — CartPole Simulator (GUI)](#step-3--cartpole-simulator-gui)
- [Step 4 — hls4ml Conversion](#step-4--conversion-of-neural-network-controller-using-hls4ml)
- [Step 5 — Testing on PC / Software Control (Local Only)](#step-5--testing-model-on-pc--running-model-via-pcsoftware-to-control-cartpole)
- [Step 6 — Implementation (Vivado/Vitis)](#step-6--implementation)
- [Step 7 — SD Card + FPGA Boot](#step-7--load-image-on-sd-card-and-onto-fpga) -->


## Step 0 — Notebook prerequisites 

- Please make sure you have Conda installed
    - Refer to [this article](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html) for instructions on setting it up
- Ensure you have access to these programs
    - Vivado 2020.1 
    - Vitis 2020.1


## Step 1 — Environment Setup

### Conda environment
- In your terminal do these
```bash
conda create -n physical_cartpole python=3.9
conda activate physical_cartpole
```
### Install Packages
- These are just the necessary packages for training, we will install the GUI/Simulation packages later
<!-- these are the gui packages: # These need PyQt6 which I dont have rn (for the GUI)
## -e ./Driver/CartPoleSimulation/SI_Toolkit/
## %pip install watchdog pydot graphviz PyQt6 -->
**Lab server note:** if your environment is pre-configured, you can skip this step.


In [ ]:
%pip install -r requirements.txt 

## Step 2 — Training Neural Network Controller

### Dataset location
- The Seed dataset is initially located at: `./Experiment-1` in the root directory which contains:
    - recorded trajectories (CSV)
    - a known good model configuration

### 2.1 Choose experiment + model name

- `CARTPOLE_EXPERIMENT_NAME` (default: `Experiment-1`)
- `CARTPOLE_NET_NAME` (default shown below)


In [11]:
import os, shutil
from pathlib import Path

# Root of the git repository (assumes notebook is inside the repo)
REPO = Path.cwd().resolve()
WORKSPACE = REPO / "Driver" / "CartPoleSimulation" / "SI_Toolkit_ASF" / "Experiments"
SI_ASF = REPO / "Driver" / "CartPoleSimulation" / "SI_Toolkit_ASF"


# Experiment name (override via env var if provided)
EXPERIMENT_NAME = os.environ.get("CARTPOLE_EXPERIMENT_NAME", "Experiment-1_TEST")

# Neural network model name to use
NET_NAME = os.environ.get("CARTPOLE_NET_NAME", "Dense-7IN-32H1-32H2-1OUT-0")

# Path to seed (template) experiment shipped with the repo
SEED_EXPERIMENT = REPO / "Experiment-1"

# Path to active experiment workspace used by SI_Toolkit
ACTIVE_EXPERIMENT = WORKSPACE / EXPERIMENT_NAME

# Echo resolved paths and selected model for verification
print("SEED_EXPERIMENT:", SEED_EXPERIMENT)
print("ACTIVE_EXPERIMENT:", ACTIVE_EXPERIMENT)
print("NET_NAME:", NET_NAME)

SEED_EXPERIMENT: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Experiment-1
ACTIVE_EXPERIMENT: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1_TEST
NET_NAME: Dense-7IN-32H1-32H2-1OUT-0


### 2.2 Move/copy seed experiment into the SI_Toolkit experiments folder

This shell scripts effectively ensure the experiment lives under `SI_Toolkit_ASF/Experiments/`.
We do a safe copy **only if missing**, to avoid overwriting trained artifacts.


In [12]:
import shutil

if not ACTIVE_EXPERIMENT.exists():
    shutil.copytree(SEED_EXPERIMENT, ACTIVE_EXPERIMENT)
    print("Copied ./Experiment-1 →", ACTIVE_EXPERIMENT)
else:
    print("Active experiment already exists: not overwriting")


Copied ./Experiment-1 → /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1_TEST


Now there are **two paths**:

### 2.3 Path A: Use precomputed model

- Skip to the Simulation (Step 3) 


### 2.4 Path B: Train the neural network
- Follow the next steps


### 2.5 Training Configuration (config_training.yml)

Run this cell to see the current training configurations in:
`Driver/CartPoleSimulation/SI_Toolkit_ASF/config_training.yml`

In [13]:
import yaml
from pprint import pprint

with open("../physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_training.yml", "r") as f:
    cfg = yaml.safe_load(f)

pprint(cfg)

{'CONFIG_SERIES_MODIFICATION': {'MODE': 'train_for_random_vertical_angle_shift',
                                'NOISE_LEVEL': {'FEATURES': 0.1}},
 'FILTERS': None,
 'PRUNING': {'ACTIVATED': False,
             'PRUNING_PARAMS': {'PRUNING_SCHEDULE': 'CONSTANT_SPARSITY'},
             'PRUNING_SCHEDULES': {'CONSTANT_SPARSITY': {'begin_step_in_epochs': 1.0,
                                                         'end_step_in_training_fraction': 1.0,
                                                         'frequency_per_epoch': 100.0,
                                                         'target_sparsity': 0.75,
                                                         'target_sparsity_last_layer': 0.0},
                                   'POLYNOMIAL_DECAY': {'begin_step_in_epochs': 1.0,
                                                        'end_step_in_training_fraction': 0.8,
                                                        'final_sparsity': 0.75,
                         

Below is an **optional** cell that updates the yamls experiment path if you changed it, leaving your hyperparameters intact.

In [14]:
import yaml

cfg_path = SI_ASF / "config_training.yml"
cfg = yaml.safe_load(cfg_path.read_text())

def set_experiment_path(cfg_obj, new_path: str) -> bool:
    if isinstance(cfg_obj, dict):
        if "paths" in cfg_obj and isinstance(cfg_obj["paths"], dict) and "path_to_experiment" in cfg_obj["paths"]:
            cfg_obj["paths"]["path_to_experiment"] = new_path
            return True
        if "PATH_TO_EXPERIMENT" in cfg_obj:
            cfg_obj["PATH_TO_EXPERIMENT"] = new_path
            return True
    return False

ok = set_experiment_path(cfg, str(ACTIVE_EXPERIMENT))
if ok:
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("Updated config_training.yml to use:", ACTIVE_EXPERIMENT)
else:
    print("Could not locate experiment path key; set it manually to:", ACTIVE_EXPERIMENT)

Updated config_training.yml to use: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1_TEST


### 2.6 Data normalization

Neural networks train much more reliably when each feature is on a comparable numeric scale.

In this project, **normalization statistics are computed from the `Train/` CSVs** inside the active experiment folder.  
Those statistics are saved to `NormalizationInfo/` and then reused consistently for:

- training (inputs/targets are normalized the same way every run)
- simulation inference (the controller sees the same scaling)
- FPGA/HLS deployment (the firmware uses the same vectors)

The function that does this is `SI_Toolkit.load_and_normalize.calculate_normalization_info(...)`.
